<a href="https://colab.research.google.com/github/Jionary/Repository-IA/blob/main/ETL_airbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import altair as alt

In [ ]:
# Configuración del estilo visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
#Deshabilitar el limite de 5000 filas de Altair
alt.data_transformers.enable('default', max_rows = None)
print("Librerías importadas")

In [ ]:
#Carga y limpieza de datos
url = "https://data.insideairbnb.com/mexico/df/mexico-city/2026-06-15/data/listings.csv.gz"
try:
  df = pl.read_csv(url)
  print("Dataset cargado exitosamente")
except Exception as e:
  print(f"Error al cargar el dataset: {e}")

#Limpieza inicial
if not df.is_empty():
  df = df.with_columns(
      pl.col("price")
      .str.replace_all(r"\$", "")
      .str.replace_all(r",", "")
      .cast(pl.Float64, strict=False)
      .alias("price"),
  ).drop_nulls(
      subset = ["price"]
  ).filter(
      pl.col("price") > 0
  )
print("Limpieza inicial realizada")
display(df.head())


In [ ]:
# Analísis básico
print("1. ¿Cuál es el precio proemdio por noche?")
avg_price = df.select(pl.col('price').mean()).item()
print(f"El precio promedio por noche es: ${avg_price}")
print("2. ¿Cuales son los tipos de alojamiento?")
room_types = df.group_by('room_type').agg(pl.len().alias('count')).sort('count', descending=True)
print(room_types)
print("3. ¿Cuales son las 10 alcaldías con mas alojamientos?")
top_10_neighbourhoods = df.group_by('neighbourhood_cleansed').agg(pl.len()
.alias('count')).sort('count', descending=True).head(16)
print(top_10_neighbourhoods)
print("4. ¿Quienes son los anfitriones con mas alojamientos?")
top_10_host = df.group_by('host_name').agg(pl.len().alias('count')).sort('count', descending=True).head(10)
print(top_10_host)



In [ ]:
#Visuaizaciones
price_to_plot_df = df.filter(pl.col('price') < df.select(pl.col('price').quantile(0.95)).item()).to_pandas()

#Usando altair
chart_hist = alt.Chart(price_to_plot_df).mark_bar().encode(
    alt.X("price:Q", bin=alt.Bin(maxbins = 50), title= "Precio por noche"),
    alt.Y("count()", title = "Frecuencia"),
    tooltip = [alt.Tooltip("count()", title="Frecuencia"), alt.Tooltip("price:Q", bin=True, title="Rango de precio")]
).properties(
    title = "Distribución de precios por noche",
    width=700,
    height=400
)

chart_hist.show()

In [ ]:
#Graficar el top de alcaldias
top_alcaldias_pd = top_10_neighbourhoods.to_pandas()

fig_hoods = px.bar(
    top_alcaldias_pd,
    y='neighbourhood_cleansed',
    x='count',
    title='Top 10 de alcaldías con más alojamientos',
    orientation="h",
    labels={
        "count": "Número de alojamientos",
        "neighbourhood_cleansed": "Alcaldía"
    },
    color='count',
    color_continuous_scale=px.colors.sequential.Viridis
    )
fig_hoods.show()

In [ ]:
#Mapa gegráfico de Precios
df_sample_for_plot = df.filter(pl.col('price') < df.select(pl.col('price').quantile(0.95)).item()).sample(n=5000, seed=42).to_pandas()

#Versión con plotly Express (Mapa interactivo)
#Nota plotly usa mapbox para los mapas
fig_map = px.scatter_mapbox(
    df_sample_for_plot,
    lat='latitude',
    lon='longitude',
    color="price",
    size="price",
    color_continuous_scale=px.colors.sequential.Viridis,
    size_max=15,
    zoom=10,
    mapbox_style="carto-positron",
    hover_name="name",
    hover_data={"neighbourhood_cleansed": True, "price": ":$.2f"}
)
fig_map.update_layout(
    title="Mapa interactivo de precioes de Airbnb en CDMX",
    legend_title_text="Precio (MXN)"
)
fig_map.show()